# ⚖️ Etapa 2: Asistente Híbrido de Contratación Pública (RAG Normativo + SQL)
**Observatorio Ciudadano de Contratación Pública (SECOP II)**

En esta etapa procesamos el corpus normativo colombiano (Leyes 80/1150/1712, Decreto 1082 y Guías CCE), evaluamos la estrategia de fragmentación (chunking), generamos representaciones vectoriales en Qdrant Cloud / Parquet local, y conectamos las cifras reales de SECOP II con el marco jurídico en un asistente híbrido con evaluación de precisión y tiempos.

In [ ]:
# Recarga automática de módulos y configuración de rutas
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from observatorio.config import SQLITE_DB_PATH, QDRANT_COLLECTION_NAME
from observatorio.data.database import ejecutar_consulta_segura

print("✅ Entorno RAG configurado.")
print(f"Colección Qdrant : {QDRANT_COLLECTION_NAME}")
print(f"Base de datos SQL: {SQLITE_DB_PATH}")


---
# 🧠 PARTE 2: Asistente Híbrido de Contratación Pública (RAG Normativo + SQL)

## 2.1 Construyendo la Biblioteca Jurídica y Técnica
Un ciudadano no solo necesita saber *cuánto* se contrató, sino *qué dice la norma* al respecto. Aquí reunimos y estructuramos el marco regulatorio esencial de la contratación pública en Colombia (**Ley 80 de 1993**, **Ley 1150 de 2007**, **Decreto 1082 de 2015**, **Circular Externa Única** y los manuales/diccionarios de **Colombia Compra Eficiente**), organizando cada documento con su fuente, fecha y artículo para poder citarlo con precisión.

In [31]:
# TODO: Lectura de archivos Markdown del corpus normativo y extracción/asignación de metadatos
from observatorio.config import CORPUS_DIR, CORPUS_MARKDOWN_DIR
from observatorio.rag.chunking import ProcesadorCorpusMarkdown

procesador = ProcesadorCorpusMarkdown()

# Conversor de PDF a Markdown (convierte todos los PDFs en el directorio `CORPUS_DIR` a Markdown y los guarda en `CORPUS_MARKDOWN_DIR`)
corpus = procesador.procesar_directorio_corpus(
    directorio_raiz=CORPUS_DIR,
    directorio_salida_md=CORPUS_MARKDOWN_DIR,
)

## 2.2 Cómo Dividir la Ley sin Perder el Contexto (Chunking)
Si partimos un artículo de ley en pedazos demasiado pequeños, se pierde el sentido jurídico; si dejamos bloques demasiado largos, la búsqueda pierde precisión. En esta sección ponemos a prueba distintas estrategias de fragmentación respetando la estructura por artículos para elegir el equilibrio ideal.

In [32]:
# TODO: Configurar al menos 3 tamaños en RecursiveCharacterTextSplitter
# TODO: Construir tabla comparativa de métricas de chunking y seleccionar el tamaño definitivo
from observatorio.rag.chunking import EvaluadorChunking, FragmentadorNormativo

evaluador = EvaluadorChunking()
fragmentador = FragmentadorNormativo()

# 1. Comparación de fragmentación recursiva por tamaños (al menos 3 configuraciones)
df_comparacion_recursivo = evaluador.comparar_recursivo(corpus)
print("📊 Comparación de fragmentación recursiva:")
display(df_comparacion_recursivo)

# 2. Comparación de fragmentación semántica por percentiles (usar_local=True para evitar consumo de API de Gemini)
df_comparacion_semantico = evaluador.comparar_semantico(
    corpus[:3], percentiles_a_probar=[90, 95, 99], usar_local=True
)
print("
📊 Comparación de fragmentación semántica:")
display(df_comparacion_semantico)

# 3. Fragmentación definitiva del corpus con la estrategia seleccionada (recursiva equilibrada)
fragmentos = fragmentador.fragmentar_corpus(
    corpus, metodo="recursivo", chunk_size=1000, chunk_overlap=150
)
print(f"
📦 Fragmentos generados: {len(fragmentos):,}")

⏳ Cuota de API alcanzada (429). Esperando 35s para reintentar automáticamente (intento 1/5)...
⏳ Cuota de API alcanzada (429). Esperando 35s para reintentar automáticamente (intento 2/5)...
⏳ Cuota de API alcanzada (429). Esperando 35s para reintentar automáticamente (intento 3/5)...
⏳ Cuota de API alcanzada (429). Esperando 35s para reintentar automáticamente (intento 4/5)...
⏳ Cuota de API alcanzada (429). Esperando 35s para reintentar automáticamente (intento 5/5)...


IndexError: list index out of range

## 2.3 Indexación Semántica en la Nube (`Qdrant Cloud` + `Gemini Embeddings`)
Transformamos cada fragmento normativo en representaciones vectoriales utilizando **Gemini Embedding 2** y los almacenamos en una base vectorial en la nube (**Qdrant Cloud**). De esta forma, el sistema puede encontrar el artículo pertinente aunque el usuario formule su pregunta con palabras cotidianas muy distintas al texto legal.

In [ ]:
# TODO: Inicializar cliente Qdrant Cloud y crear colección con VectorParams(size=768, distance=Distance.COSINE)
# TODO: Generar embeddings por lotes con Gemini Embedding 2, guardar respaldo local y hacer upsert en Qdrant
import importlib
import observatorio.rag.vectorstore as vs

importlib.reload(vs)
from observatorio.rag.vectorstore import IndexadorQdrant, EMBEDDINGS_BACKUP_PATH

# Inicializar indexador (puedes omitir usar_local para intentar con Gemini y fallback automático a local)
indexador = IndexadorQdrant(usar_local=True)

total_en_qdrant = indexador.indexar_fragmentos(fragmentos)
print(f"✅ Total de puntos vectoriales indexados en Qdrant: {total_en_qdrant:,}")

assert total_en_qdrant == len(
    fragmentos
), f"Discrepancia de conteo: Qdrant ({total_en_qdrant}) != Fragmentos ({len(fragmentos)})"
print(
    f"✔️ Verificado: {total_en_qdrant:,} puntos en Qdrant == {len(fragmentos):,} fragmentos"
)
print(f"Respaldo de vectores: {EMBEDDINGS_BACKUP_PATH}")

🔍 Inicializando IndexadorQdrant en colección 'normativa_contratacion'.
   Qdrant URL: https://c0be2581-0382-42bc-82fe-47333af56596.us-east-2-0.aws.cloud.qdrant.io
   Modo inicial de embeddings: LOCAL (sentence-transformers/paraphrase-multilingual-mpnet-base-v2)
ℹ️ La colección 'normativa_contratacion' ya se encuentra activa en Qdrant Cloud.
📂 Respaldo local detectado: 4,254 vectores cargados desde embeddings_normativa.parquet.
✅ Todos los 4,256 fragmentos ya cuentan con embeddings en disco.
☁️ Subiendo 4,256 puntos a Qdrant Cloud en colección 'normativa_contratacion'...
✅ Indexación finalizada: 4,256 puntos activos en Qdrant Cloud.


ValueError: Cannot specify ',' with '\x20'.

## 2.4 Conectando las Cifras (`SQL`) con las Normas (`RAG`)
Aquí unimos ambos mundos: diseñamos el buscador semántico sobre la normativa y el motor de consultas de solo lectura sobre la base de contratos. Así, el asistente puede decidir cuándo responder con estadísticas de la base de datos, cuándo citar un artículo de ley o cuándo combinar ambos en una sola respuesta.

In [ ]:
# TODO: Definir función de recuperación semántica Top-k sobre Qdrant con filtro opcional por documento
# TODO: Definir herramienta de consulta SQL de solo lectura (validación de SELECT único y límite de filas)

df_resultados = indexador.buscar_similares(
    "¿Cuál es la duración máxima permitida para un contrato de obras públicas según la ley?",
    top_k=3,
)
display(df_resultados)

# Con filtro por documento
display(
    indexador.buscar_similares(
        "principios de la contratación estatal",
        top_k=3,
        filtro_documento="Ley 80 de 1993",
    )
)

display(
    indexador.buscar_similares(
        "cuáles son los principios de la contratación pública",
        top_k=3,
        filtro_documento="Ley 1150 de 2007",
    )
)

# 1. Consulta SQL válida (ej: top 5 modalidades con mayor cuantía)
sql_ejemplo = """
SELECT modalidad_de_contratacion, COUNT(*) AS total_contratos, ROUND(SUM(valor_del_contrato), 2) AS valor_total
FROM contratos
GROUP BY modalidad_de_contratacion
ORDER BY valor_total DESC
"""
df_cifras, sql_ejecutado = ejecutar_consulta_segura(sql_ejemplo)
print(f"SQL ejecutado con seguridad: {sql_ejecutado}")
display(df_cifras)
# 2. Demostración de protección (bloquea inyecciones o mutaciones)
try:
    ejecutar_consulta_segura("DELETE FROM contratos WHERE valor_del_contrato = 0")
except ValueError as e:
    print(f"🛡️ Bloqueo de seguridad verificado: {e}")

,rank,score_coseno,documento,articulo,tipo_fuente,fecha,texto
0,1,0.6953,Decreto 1082 de 2015,Artículo 34 d,Decreto Reglamentario,26 de mayo de 2015,"7. En cumplimiento de lo previsto en el artículo 34 de la Ley 1508 de 2012, ..."
1,2,0.6549,Decreto 1082 de 2015,Artículo 2,Decreto Reglamentario,26 de mayo de 2015,Capacidad Residual del Proceso de Contratación = Presupuesto oﬁcial estimado...
2,3,0.6502,Decreto 1082 de 2015,Artículo 2,Decreto Reglamentario,26 de mayo de 2015,ARTÍCULO 2.2.1.1.1.6.4. Capacidad Residual. El interesado en celebrar contra...


,rank,score_coseno,documento,articulo,tipo_fuente,fecha,texto
0,1,0.0752,Ley 80 de 1993,Artículo 224 d,Ley de la República,28 de octubre de 1993,Notas de Vigencia\nJurisprudencia Vigencia\n\n3o. <Numeral modificado por el...
1,2,0.0601,Ley 80 de 1993,Artículo 1 d,Ley de la República,28 de octubre de 1993,de selección se llamará concurso y se efectuará también mediante invitación ...
2,3,0.0505,Ley 80 de 1993,Artículo 1 d,Ley de la República,28 de octubre de 1993,Notas de Vigencia\n\nPARÁGRAFO 3o. <Parágrafo adicionado por el artícu...


,rank,score_coseno,documento,articulo,tipo_fuente,fecha,texto
0,1,0.0512,Ley 1150 de 2007,Artículo 9,Ley de la República,16 de julio de 2007,ARTÍCULO 9o. DE LA ADJUDICACIÓN. En el evento previsto en el artículo 273 de...
1,2,0.0459,Ley 1150 de 2007,Artículo 2,Ley de la República,16 de julio de 2007,Para las entidades que tengan un presupuesto anual superior o igua...
2,3,0.0433,Ley 1150 de 2007,Artículo 54 d,Ley de la República,16 de julio de 2007,siguiente:> Corresponde a Ia modalidad prevista para Ia selección d...


SQL ejecutado con seguridad: SELECT modalidad_de_contratacion, COUNT(*) AS total_contratos, ROUND(SUM(valor_del_contrato), 2) AS valor_total
FROM contratos
GROUP BY modalidad_de_contratacion
ORDER BY valor_total DESC LIMIT 100


,modalidad_de_contratacion,total_contratos,valor_total
0,Contratación directa,46449,1.452819e+12
1,Contratación régimen especial,2206,4.464546e+11
2,Licitación pública,41,2.997095e+11
3,Contratación Directa (con ofertas),159,2.135236e+11
4,Contratación régimen especial (con ofertas),137,1.050171e+11
5,Seleccion Abreviada Menor Cuantia Sin Manifestacion Interes,4,8.428415e+10
6,Selección abreviada subasta inversa,47,5.816516e+10
7,Licitación pública Obra Publica,13,5.597588e+10
8,Selección Abreviada de Menor Cuantía,133,5.169360e+10
9,Mínima cuantía,786,2.739679e+10


🛡️ Bloqueo de seguridad verificado: Operación bloqueada: solo se permiten consultas SELECT de lectura.


## 2.5 Poniendo a Prueba al Asistente: Precisión, Tiempos y Límites
Para asegurarnos de que el asistente sea confiable en escenarios reales, lo evaluamos con un banco de preguntas directas, preguntas en lenguaje coloquial, consultas que mezclan cifras con legislación y preguntas trampa sobre temas fuera del sistema (verificando que reconozca cuando no tiene la información en lugar de inventar respuestas).

In [ ]:
# Evaluación automatizada del Asistente Híbrido sobre el banco de 12 preguntas de validación
from observatorio.rag.assistant import AsistenteHibrido
from observatorio.rag.evaluation import ejecutar_evaluacion_completa

# 1. Instanciar asistente en modo local
asistente = AsistenteHibrido(indexador_vectorial=indexador, usar_local=True)

# 2. Ejecutar protocolo de evaluación (protocolo estandarizado)
df_evaluacion, df_resumen = ejecutar_evaluacion_completa(asistente)

# 3. Métricas consolidadas (Acierto de enrutamiento, fuentes, alucinación y latencia)
print("📊 Resumen Global de Evaluación:")
display(df_resumen)

# 4. Tabla detallada por pregunta y categoría
print("\n🔎 Detalle de Evaluación por Pregunta:")
cols_ver = ["id", "categoria", "tipo_esperado", "tipo_obtenido", "acierto_tipo", "acierto_fuente", "score_max", "latencia_s", "sql_ejecutado", "fuentes_doc"]
display(df_evaluacion[cols_ver])

🤖 AsistenteHibrido inicializado en modo: 100% LOCAL (sin Gemini).
Tipo: cuantitativa | Modo: local | Latencia: 0.969
¿Cuál es el valor promedio de los contratos de servicios profesionales?
### 📊 Cifras del Observatorio (SECOP II)
Consulta analítica ejecutada: `SELECT tipo_de_contrato, COUNT(*) AS total_contratos, ROUND(SUM(valor_del_contrato), 2) AS valor_total FROM contratos WHERE 1=1 GROUP BY tipo_de_contrato ORDER BY total_contratos DESC LIMIT 10`

| tipo_de_contrato | total_contratos | valor_total |
| --- | --- | --- |
| Prestación de servicios | 47624 | 1428027478474.84 |
| Decreto 092 de 2017 | 779 | 88342895244.96 |
| Otro | 667 | 767334372727.01 |
| Suministros | 334 | 41318260718.17 |
| Arrendamiento de inmuebles | 192 | 26208963221.98 |
| Compraventa | 146 | 19304949325.51 |
| Obra | 87 | 195126451334.06 |
| Comodato | 60 | 477556631.0 |
| Seguros | 36 | 60910223379.0 |
| Consultoría | 24 | 6326604710.44 |

Tipo: cuantitativa | Modo: local | Latencia: 0.079
¿Cuáles son las 5 